# Declaration of Originality

**School of Informatics & IT**
<br/>**Diploma in Applied Artificial Intelligence**
<br/>**Machine Learning for Developers (CAI2C08)**
<br/>**AY2026/2027 April Semester**
<br/>**Program Codes**

* Student Name: Grishm Chandru Mirpuri



**Declaration of Originality**
* I am the originator of this work, and I have appropriately acknowledged all other original sources used as my references for this work.
* I understand that Plagiarism is the act of taking and using the whole or any part of another person’s work, including work generated by AI, and presenting it as my own.
* I understand that Plagiarism is an academic offence and if I am found to have committed or abetted the offence of plagiarism in relation to this submitted work, disciplinary action will be enforced.

# Libraries

In [127]:
# Import libraries

# Data Handling
import pandas as pd
import numpy as np

# Data splitting
from sklearn.model_selection import train_test_split

# Scaling
from sklearn.preprocessing import StandardScaler

# Machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

# Model evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score

# Hyperparameter tuning
from sklearn.model_selection import RandomizedSearchCV

# Save final model
import joblib

# 1. Business Understanding
The goal of this project is to build a machine learning model that can predict whether a hotel booking is likely to be cancelled.
Hotel booking cancellations can affect hotel revenue and room planning. If hotels can identify bookings with higher cancellation risks earlier, staff can take suitable actions such as sending confirmation reminders, reviewing high-risk bookings or planning room availability more carefully.
This project is a binary classification model because the model predicts one of two outcomes:
- Canceled
- Not_Canceled

The target column is "booking status"
The final solution will be deployed as a Streamlit web application where users can enter booking details and recieve a cancelling prediction, cancellation probability, risk level and simple business recommendation.

# 2. Data Understanding

## 2.1 Load dataset

In [128]:
# Read *.csv file into pandas DataFrame
df = pd.read_csv("../data/hotel_bookings.csv")

# Display the first 5 rows of the DataFrame
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


The dataset was loaded using pandas. The first 5 rows were displayed to confirm that the file was imported correctly.

## 2.2 Basic Dataset Checks

In [129]:
# Check the number of rows and columns in the DataFrame
df.shape

(119390, 32)

The dataset has 119,390 rows and 32 columns. This is great because it gives us a huge sample size - way more than the minimum 200 records needed for the project. With this much data, we can safely train advanced models like Random Forest without worrying about running out of data or overfitting.

In [130]:
# Display all column names in the DataFrame
df.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'reservation_status', 'reservation_status_date'],
      dtype='object')

I checked the columns and confirmed that `is_canceled` is the target variable. Getting this down first is important because it tells me exactly what I are trying to predict. It helps split the input features from the target before doing any scaling or model training."

In [131]:
# Display basic information about the DataFrame
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

The dataset information was checked to understand the data types and non null values before cleaning.

## 2.3 Summary Statistics and Missing Values

In [132]:
# Understand the type of variable for each column
# Display summary statistics for numerical columns
df.describe().T

,count,mean,std,min,25%,50%,75%,max
is_canceled,119390.0,0.370416,0.482918,0.00,0.00,0.000,1.0,1.0
lead_time,119390.0,104.011416,106.863097,0.00,18.00,69.000,160.0,737.0
arrival_date_year,119390.0,2016.156554,0.707476,2015.00,2016.00,2016.000,2017.0,2017.0
arrival_date_week_number,119390.0,27.165173,13.605138,1.00,16.00,28.000,38.0,53.0
arrival_date_day_of_month,119390.0,15.798241,8.780829,1.00,8.00,16.000,23.0,31.0
stays_in_weekend_nights,119390.0,0.927599,0.998613,0.00,0.00,1.000,2.0,19.0
stays_in_week_nights,119390.0,2.500302,1.908286,0.00,1.00,2.000,3.0,50.0
adults,119390.0,1.856403,0.579261,0.00,2.00,2.000,2.0,55.0
children,119386.0,0.103890,0.398561,0.00,0.00,0.000,0.0,10.0
babies,119390.0,0.007949,0.097436,0.00,0.00,0.000,0.0,10.0


In [133]:
df["customer_type"].value_counts()

customer_type
Transient          89613
Transient-Party    25124
Contract            4076
Group                577
Name: count, dtype: int64

The summary statistics were checked to understand the range, average and spread of the numerical columns. This helps identify possible numerical values before data cleaning and modelling.

In [134]:
# Display summary statistics for categorical columns
df.describe(include='object').T

,count,unique,top,freq
hotel,119390,2,City Hotel,79330
arrival_date_month,119390,12,August,13877
meal,119390,5,BB,92310
country,118902,177,PRT,48590
market_segment,119390,8,Online TA,56477
distribution_channel,119390,5,TA/TO,97870
reserved_room_type,119390,10,A,85994
assigned_room_type,119390,12,A,74053
deposit_type,119390,3,No Deposit,104641
customer_type,119390,4,Transient,89613


The categorical summary was checked to understand to check the main categories of the dataset and whether any columns may need cleaning before encoding.

In [135]:
# Check for missing values in each column
df.isnull().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

Missing values were checked because machine learning models cannot handle missing values directly. Any missing values (`children`, `country`, `agent` and `company`) found will need to be handled during data cleaning.

In [136]:
# Check for duplicate rows in the DataFrame
df.duplicated().sum()

np.int64(31994)

Duplicate rows were checked as part of data understanding. The decision on whether to remove them will be made during data cleaning.

## 2.3 Exploratory Data Analysis using Summary Tables

## 2.4 Target Variable Distribution

In [137]:
# Understanding distribution of target
target_count = df["is_canceled"].value_counts()
target_percentage = df["is_canceled"].value_counts(normalize=True) * 100
print("Target variable distribution:")
print(target_count)
print("\nTarget variable distribution (percentage):")
print(target_percentage)


Target variable distribution:
is_canceled
0    75166
1    44224
Name: count, dtype: int64

Target variable distribution (percentage):
is_canceled
0    62.958372
1    37.041628
Name: proportion, dtype: float64


The target distribution shows the number and percentage of cancelled and not cancelled bookings. This helps to check whether the classification problem is balanced or imbalanced. Since cancellation prediction is the business focus, accuracy alone may not be enough. Precision, recall and f1-score will also be considered.

## 2.5 EDA Approach

## 2.5.1 Correlation with Target

In [194]:
# Check correlation between numerical features and target variable

corr_with_target = df.corr(numeric_only=True)["is_canceled"].sort_values(ascending=False)

corr_table = pd.DataFrame({
    "Correlation with is_canceled": corr_with_target.round(3)
})

corr_table

,Correlation with is_canceled
is_canceled,1.000
lead_time,0.293
previous_cancellations,0.110
adults,0.060
days_in_waiting_list,0.054
adr,0.048
total_guests,0.047
stays_in_week_nights,0.025
total_nights,0.018
arrival_date_year,0.017


I checked the correlation values to see which numerical columns have the strongest link to `is_canceled`. The columns that stood out the most were `lead_time`, `total_of_special_requests`, and `required_car_parking_spaces` because their numbers were furthest away from 0. This makes total sense from a business perspective too - how early someone books, how many special requests they have, and whether they need parking are all strong hints about how committed they are to the trip. These features will be explored further using summary tables.

## 2.5.2 EDA on Selected Numerical Features

In [195]:
# Create lead time groups to understand cancellation distribution by lead time
df["lead_time_group"] = pd.cut(df["lead_time"], bins=[-1, 30, 90, 180, 365, np.inf], labels=["0-30", "31-90", "91-180", "181-365", "366+"])
cancellation_by_lead_time = df.groupby("lead_time_group")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by lead time group:")
print(cancellation_by_lead_time)


Cancellation distribution by lead time group:
is_canceled             0         1
lead_time_group                    
0-30             0.814370  0.185630
31-90            0.623016  0.376984
91-180           0.552895  0.447105
181-365          0.445460  0.554540
366+             0.323380  0.676620


C:\Users\grish\AppData\Local\Temp\ipykernel_28468\3813649122.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cancellation_by_lead_time = df.groupby("lead_time_group")["is_canceled"].value_counts(normalize=True).unstack()


Looking at the summary table, there's a clear trend with lead time. Bookings made months in advance have a much higher cancellation rate than those booked closer to the arrival date. This makes `lead_time` a really important feature because it gives the models a strong, steady pattern to learn from.

In [196]:
# Cancellation rate by number of special requests
cancellation_by_special_requests = df.groupby("total_of_special_requests")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by number of special requests:")
print(cancellation_by_special_requests)


Cancellation distribution by number of special requests:
is_canceled                       0         1
total_of_special_requests                    
0                          0.522796  0.477204
1                          0.779751  0.220249
2                          0.779011  0.220989
3                          0.821386  0.178614
4                          0.894118  0.105882
5                          0.950000  0.050000


The data here shows that the more special requests a guest makes, the less likely they are to cancel. Operationally, this means guests who put in extra effort to customize their stay are way more invested in their booking. This feature is definitely a useful keep for the input features because it helps the model flag low-risk vs high-risk bookings easily.

In [ ]:
# Count bookings by required car parking spaces
parking_space_counts = df["required_car_parking_spaces"].value_counts()
print("\nBooking counts by required car parking spaces:")
print(parking_space_counts)


Booking counts by required car parking spaces:
required_car_parking_spaces
0    111974
1      7383
2        28
3         3
8         2
Name: count, dtype: int64


In [ ]:
# Cancelation rate by required car parking spaces
cancellation_by_parking_spaces = df.groupby("required_car_parking_spaces")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by required car parking spaces:")
print(cancellation_by_parking_spaces)


Cancellation distribution by required car parking spaces:
is_canceled                         0         1
required_car_parking_spaces                    
0                            0.605051  0.394949
1                            1.000000       NaN
2                            1.000000       NaN
3                            1.000000       NaN
8                            1.000000       NaN


The count table shows that most bookings required either 0 or 1 car parking space. Very few bookings required 2, 3 or 8 parking spaces.

Although bookings with parking spaces appear to have lower cancellation rates, some categories have very small counts. This means the cancellation percentages for those categories may not be reliable.
Therefore, `required_car_parking_spaces` will not be selected as a main feature for detailed EDA and for modelling later.

## 2.5.3 EDA on Selected Categorical Features

In [143]:
# Count bookings by hotel type
hotel_type_counts = df["hotel"].value_counts()
print("\nBooking counts by hotel type:")
print(hotel_type_counts)


Booking counts by hotel type:
hotel
City Hotel      79330
Resort Hotel    40060
Name: count, dtype: int64


In [144]:
# Cancellation rate by hotel type
cancellation_by_hotel = df.groupby("hotel")["is_canceled"].value_counts(normalize=True).unstack()
print("Cancellation distribution by hotel type:")
print(cancellation_by_hotel)

Cancellation distribution by hotel type:
is_canceled          0         1
hotel                           
City Hotel    0.582730  0.417270
Resort Hotel  0.722366  0.277634


The count table shows that the dataset contains bookings from both city hotels and resort hotels.
The cancelation rate differs between the two hotel types, suggesting that `hotel` may be useful as a categorical feature for modelling after encoding. This also makes business sense because city hotels and resort hotels may have different customer behavior and cancelation patterns.

In [145]:
# Count bookings by deposit type
deposit_type_counts = df["deposit_type"].value_counts()
print("\nBooking counts by deposit type:")
print(deposit_type_counts)


Booking counts by deposit type:
deposit_type
No Deposit    104641
Non Refund     14587
Refundable       162
Name: count, dtype: int64


The count of bookings by deposit type was checked first to understand how common each deposit category is.

In [146]:
# Cancellation rate by deposit type
cancellation_by_deposit_type = df.groupby("deposit_type")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by deposit type:")
print(cancellation_by_deposit_type)


Cancellation distribution by deposit type:
is_canceled          0         1
deposit_type                    
No Deposit    0.716230  0.283770
Non Refund    0.006376  0.993624
Refundable    0.777778  0.222222


I keep `deposit_type` because the count tables showed it has massive predictive power. Bookings marked as 'Non Refund' have a near-100% cancellation rate in this dataset. This feature gives a really strong signal to both the Baseline Logistic Regression and the tree models, so it's a must-have for the input features.

In [147]:
# Count bookings by market segment
market_segment_counts = df["market_segment"].value_counts()
print("\nBooking counts by market segment:")
print(market_segment_counts)


Booking counts by market segment:
market_segment
Online TA        56477
Offline TA/TO    24219
Groups           19811
Direct           12606
Corporate         5295
Complementary      743
Aviation           237
Undefined            2
Name: count, dtype: int64


In [148]:
# Cancellation rate by market segment
cancellation_by_market_segment = df.groupby("market_segment")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by market segment:")
print(cancellation_by_market_segment)


Cancellation distribution by market segment:
is_canceled            0         1
market_segment                    
Aviation        0.780591  0.219409
Complementary   0.869448  0.130552
Corporate       0.812653  0.187347
Direct          0.846581  0.153419
Groups          0.389380  0.610620
Offline TA/TO   0.656840  0.343160
Online TA       0.632789  0.367211
Undefined            NaN  1.000000


The count table shows how bookings are distributed across different market segments. This is important because categories with fewer records may give less reliable percentages.
The cancelation rate differs across market segments. This suggests that `market_segment` may be useful as a categorical feature for modelling after encoding. This also makes business sense because different booking sources or customer groups may have different cancelation behavior.

In [149]:
# Count bookings by customer type
customer_type_counts = df["customer_type"].value_counts()
print("\nBooking counts by customer type:")
print(customer_type_counts)


Booking counts by customer type:
customer_type
Transient          89613
Transient-Party    25124
Contract            4076
Group                577
Name: count, dtype: int64


In [150]:
# Cancelation rate by customer type
cancellation_by_customer_type = df.groupby("customer_type")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by customer type:")
print(cancellation_by_customer_type)


Cancellation distribution by customer type:
is_canceled             0         1
customer_type                      
Contract         0.690383  0.309617
Group            0.897747  0.102253
Transient        0.592537  0.407463
Transient-Party  0.745701  0.254299


The count table shows how bookings are distributed across different customer types. This helps check whether each customer type has enough fecords for the cancellation percentages to be meaningful.
The cancellation rate differs across customer types. This sugggsts that `customer_type` may be useful as a categorical feature for modelling after encoding. This also makes business sense because individual, group and contract-related bookings may have different cancellation behavior.

## 2.5.4 EDA using Feature-Engineered Columns

In [151]:
# Create simple feature-engineering columns based on existing columns
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"] = df["adults"] + df["children"] + df["babies"]
df["has_children"] = np.where(df["children"] + df["babies"] > 0, 1, 0)
df["has_special_request"] = np.where(df["total_of_special_requests"] > 0, 1, 0)

The engineered features were created to make the booking information easier to understand and more useful for modelling.

`total_nights` combines weekend nights and week nights into one feature that represents the full length of stay.

`total_guests` combines adults, children and babies into one feature that represents the total number of guests in the booking.

`has_children` simplifies the children and babies columns into a yes/no feature. This helps check whether family-related bookings have different cancellation behaviour.

`has_special_request` simplifies the number of special requests into a yes/no feature. This helps check whether guests who make at least one special request have different cancellation behaviour.

In [152]:
# Create total nights groups for EDA
df["total_nights_group"] = pd.cut(df["total_nights"], bins=[-1, 1, 3, 7, 14, np.inf], labels=["0-1", "2-3", "4-7", "8-14", "15+"])

# Count bookings by total nights group
booking_counts_by_total_nights_group = df.groupby("total_nights_group").size()
print("\nBooking counts by total nights group:")
print(booking_counts_by_total_nights_group)

# Cancellation rate by total nights group
cancellation_by_total_nights_group = df.groupby("total_nights_group")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by total nights group:")
print(cancellation_by_total_nights_group)


Booking counts by total nights group:
total_nights_group
0-1     21735
2-3     54719
4-7     37679
8-14     4818
15+       439
dtype: int64

Cancellation distribution by total nights group:
is_canceled                0         1
total_nights_group                    
0-1                 0.755878  0.244122
2-3                 0.570277  0.429723
4-7                 0.641047  0.358953
8-14                0.660440  0.339560
15+                 0.446469  0.553531


C:\Users\grish\AppData\Local\Temp\ipykernel_28468\2971147334.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  booking_counts_by_total_nights_group = df.groupby("total_nights_group").size()
C:\Users\grish\AppData\Local\Temp\ipykernel_28468\2971147334.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cancellation_by_total_nights_group = df.groupby("total_nights_group")["is_canceled"].value_counts(normalize=True).unstack()


The cancellation rate differs across total night groups. Short stays of 0–1 nights have a lower cancellation rate, while longer stays, especially 15+ nights, show a higher cancellation rate. This suggests that `total_nights` may be useful as an engineered feature for modelling. Combining week nights and weekend nights gives the model a single, clear number representing the full length of the stay. This makes the feature easier for the model to process and simplifies what the user has to type into the Streamlit app.

In [153]:
# Create total guests groups for EDA
df["total_guests_group"] = pd.cut(df["total_guests"], bins=[-1, 1, 2, 4, 6, np.inf], labels=["0", "1", "2", "3-4", "5+"])

# Count bookings by total guest group
booking_counts_by_total_guests_group = df.groupby("total_guests_group").size()
print("\nBooking counts by total guests group:")
print(booking_counts_by_total_guests_group)

# Cancellation rate by total guests group
cancellation_by_total_guests_group = df.groupby("total_guests_group")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by total guests group:")
print(cancellation_by_total_guests_group)


Booking counts by total guests group:
total_guests_group
0      22761
1      82048
2      14423
3-4      138
5+        16
dtype: int64

Cancellation distribution by total guests group:
is_canceled                0         1
total_guests_group                    
0                   0.710909  0.289091
1                   0.603049  0.396951
2                   0.651806  0.348194
3-4                 0.746377  0.253623
5+                  0.125000  0.875000


C:\Users\grish\AppData\Local\Temp\ipykernel_28468\2417345942.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  booking_counts_by_total_guests_group = df.groupby("total_guests_group").size()
C:\Users\grish\AppData\Local\Temp\ipykernel_28468\2417345942.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cancellation_by_total_guests_group = df.groupby("total_guests_group")["is_canceled"].value_counts(normalize=True).unstack()


The original `total_guests` values had some very small groups, especially for larger guest numbers. Therefore, the values were grouped to make the pattern easier to interpret.

The count table shows that most bookings are for 1 to 4 guests, while 5+ guest bookings are much less common. The cancellation rate differs across guest groups, suggesting that `total_guests` may be useful as an engineered feature for modelling.

Bookings with 0 guests may represent unusual or invalid records, so they will be reviewed later during data cleaning.

In [154]:
# Count bookings by whether the booking includes children or babies
children_flag_counts = df["has_children"].value_counts().sort_index()
print("Booking counts by has_children")
print(children_flag_counts)

# Cancellation rate by whether the booking includes children or babies
cancellation_by_children_flag = df.groupby("has_children")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by has_children")
print(cancellation_by_children_flag)

Booking counts by has_children
has_children
0    110058
1      9332
Name: count, dtype: int64

Cancellation distribution by has_children
is_canceled          0         1
has_children                    
0             0.627787  0.372213
1             0.650772  0.349228


The count table shows that most bookings do not include children or babies. The cancellation rate is slightly lower for bookings with children or babies, but the difference is small.

This suggests that `has_children` may not be a strong feature on its own. It can still be considered during modelling, but it should not be treated as one of the main predictors based on EDA alone.

In [155]:
# Count bookings by whether the booking has special requests
special_requests_flag_counts = df["has_special_request"].value_counts().sort_index()
print("Booking counts by has_special_request")
print(special_requests_flag_counts)

# Cancellation rate by whether the booking has special requests
cancellation_by_special_requests_flag = df.groupby("has_special_request")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by has_special_request")
print(cancellation_by_special_requests_flag)

Booking counts by has_special_request
has_special_request
0    70318
1    49072
Name: count, dtype: int64

Cancellation distribution by has_special_request
is_canceled                 0         1
has_special_request                    
0                    0.522796  0.477204
1                    0.782605  0.217395


The count table shows that there are many bookings both with and without special requests, so the comparison is meaningful.

The cancellation rate is much lower for bookings with at least one special request. Bookings without special requests have about 47.8% cancellations, while bookings with special requests have about 21.7% cancellations.

This suggests that `has_special_request` is a useful engineered feature for modelling. It may indicate stronger guest commitment because guests who make special requests may be more likely to follow through with their booking.

## 2.5.5 Summary of EDA Findings

The EDA was carried out using summary tables.

The target distribution showed that the dataset contains both cancelled and non-cancelled bookings, which are imbalanced. Since the project focuses on predicting cancellation, accuracy alone may not be enough. Precision, recall, F1-score and confusion matrix will also be used later during model evaluation.

For numerical features, correlation was used to identify features that may have a relationship with `is_canceled`. Based on the correlation results and business meaning, `lead_time` and `total_of_special_requests` were explored further. The EDA showed that cancellation rates differ across lead time groups and number of special requests, suggesting that these features may be useful for modelling.

For categorical features, `hotel`, `deposit_type`, `market_segment` and `customer_type` were explored using count tables and cancellation rate tables. The cancellation rates differed across these categories, suggesting that these features may be useful after encoding with `pd.get_dummies()`.

Feature-engineered columns were also created during EDA. `total_nights`, `total_guests`, `has_children` and `has_special_request` were explored to see whether simplified booking features could reveal useful cancellation patterns. `total_nights`, `total_guests` and `has_special_request` showed useful differences in cancellation behaviour. `has_children` showed only a small difference, so it will not be treated as a main predictor based on EDA alone.

Overall, the EDA suggests that both selected original features and selected engineered features may be useful for predicting hotel booking cancellation. The final usefulness of these features will be checked later during model training and comparison.

# 3. Data Preparation

## 3.1 Data Cleaning

Data cleaning was carried out to prepare the dataset for modelling. This includes handling missing values, reviewing duplicate rows, removing invalid records and dropping columns that may cause data leakage.

In [156]:
# Create a copy of the dataset for cleaning and preprocessing
df_model = df.copy()

A copy of the original dataset was created so that data preparation changes do not affect the original EDA dataset.

In [157]:
# Check missing values before cleaning
missing_values_before_cleaning = df_model.isnull().sum().sort_values(ascending=False).head(10)
print("Missing values before cleaning:")
print(missing_values_before_cleaning)

Missing values before cleaning:
company                      112593
agent                         16340
country                         488
children                          4
total_guests                      4
total_guests_group                4
hotel                             0
arrival_date_day_of_month         0
arrival_date_week_number          0
arrival_date_month                0
dtype: int64


The missing value check shows that `company`, `agent`, `country` and `children` contain missing values.

The columns `company`, `agent` and `country` are not part of the selected features for this model, so they will not be handled individually at this stage. They will be excluded later during feature selection.

In [158]:
# Fill missing values in children with 0 (assuming missing means no children)
df_model["children"].fillna(0, inplace=True)


C:\Users\grish\AppData\Local\Temp\ipykernel_28468\629378725.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_model["children"].fillna(0, inplace=True)


The `children` column is needed to create engineered features such as `total_guests` and `has_children`. Therefore, missing values in `children` will be filled with 0, as a missing value likely means no children were recorded for the booking.

In [159]:
# Create total_guest temporarily to identify bookings with 0 guests (which is invalid)
df_model["total_guests"] = df_model["adults"] + df_model["children"] + df_model["babies"]
# Count bookings with 0 guests
zero_guest_bookings = df_model[df_model["total_guests"] == 0].shape[0]
print(f"\nNumber of bookings with 0 guests: {zero_guest_bookings}")


Number of bookings with 0 guests: 180


In [160]:
# Remove bookings with 0 guests from the dataset
df_model = df_model[df_model["total_guests"] > 0]

In [161]:
# Check dataset size after removing bookings with 0 guests
print(f"\nDataset size after removing bookings with 0 guests: {df_model.shape}")


Dataset size after removing bookings with 0 guests: (119210, 39)


Rows with 0 total guests were removed because they are likely invalid or unusual records. Removing them helps keep the modelling dataset more realistic.

In [162]:
# Check duplicate rows in modelling dataset
duplicate_rows = df_model.duplicated().sum()
print(f"\nNumber of duplicate rows in modelling dataset: {duplicate_rows}")


Number of duplicate rows in modelling dataset: 31980


Duplicate rows were not removed because the dataset does not contain a unique booking ID. Identical rows may represent separate hotel bookings with the same characteristics, so removing them could remove valid records.

In [163]:
# Drop columns that may cause data leakage
columns_to_drop = ["reservation_status", "reservation_status_date"]
df_model = df_model.drop(columns=columns_to_drop, axis=1, errors='ignore')  # Use errors='ignore' to avoid KeyError if columns are not present  

`reservation_status` and `reservation_status_date` were dropped to prevent data leakage. Because `reservation_status` updates to 'Canceled' at the exact moment a guest cancels, keeping it in would basically give the answer away to the model. It would look like our model is getting 100% accuracy during training, but it would completely fail on the Streamlit app when handling new bookings.

## 3.2 Feature Engineering

In [164]:
# Check engineered feature columns
df_model[["total_nights", "total_guests", "has_children", "has_special_request"]].head()

,total_nights,total_guests,has_children,has_special_request
0,0,2.0,0,0
1,0,2.0,0,0
2,1,1.0,0,0
3,1,1.0,0,0
4,2,2.0,0,1


The output confirms that the engineered features are available in the modeling dataset and can be considered during feature selection.

## 3.3 Feature Selection

In [165]:
# Select final features for modelling

selected_features = [
    # Original features
    "hotel",
    "lead_time",
    "deposit_type",
    "market_segment",
    "customer_type",

    # Engineered features
    "total_nights",
    "total_guests",
    "has_children",
    "has_special_request"
]

X = df_model[selected_features]
y = df_model["is_canceled"]

`has_special_request` was selected instead of `total_of_special_requests` because the EDA showed a clear difference between bookings with no special requests and bookings with at least one special request. This simplified feature is easier to interpret and more practical for the Streamlit app, where we want to quickly identify if a booking has any special requests rather than the exact number of special requests.

In [166]:
# Check selected features
X.head()

,hotel,lead_time,deposit_type,market_segment,customer_type,total_nights,total_guests,has_children,has_special_request
0,Resort Hotel,342,No Deposit,Direct,Transient,0,2.0,0,0
1,Resort Hotel,737,No Deposit,Direct,Transient,0,2.0,0,0
2,Resort Hotel,7,No Deposit,Direct,Transient,1,1.0,0,0
3,Resort Hotel,13,No Deposit,Corporate,Transient,1,1.0,0,0
4,Resort Hotel,14,No Deposit,Online TA,Transient,2,2.0,0,1


In [167]:
# Check target values
y.value_counts()

is_canceled
0    75011
1    44199
Name: count, dtype: int64

The selected features include booking timing, hotel type, customer segment, deposit type, guest details and engineered booking features. A focussed set of features was chosen to keep the model understandable and to make the final Streamlit app practical for hotel staff to use. Features were selected based on EDA findings, business meaning and whether they can be reasonably entered by user before prediction.

## 3.4 Encoding Categorical variables

In [168]:
# Encode categorical columns using one-hot encoding
X_encoded = pd.get_dummies(X, columns=['hotel', 'deposit_type', 'market_segment', 'customer_type'], drop_first=True)

# Check encoded features
X_encoded.head()

,lead_time,total_nights,total_guests,has_children,has_special_request,hotel_Resort Hotel,deposit_type_Non Refund,deposit_type_Refundable,market_segment_Complementary,market_segment_Corporate,market_segment_Direct,market_segment_Groups,market_segment_Offline TA/TO,market_segment_Online TA,market_segment_Undefined,customer_type_Group,customer_type_Transient,customer_type_Transient-Party
0,342,0,2.0,0,0,True,False,False,False,False,True,False,False,False,False,False,True,False
1,737,0,2.0,0,0,True,False,False,False,False,True,False,False,False,False,False,True,False
2,7,1,1.0,0,0,True,False,False,False,False,True,False,False,False,False,False,True,False
3,13,1,1.0,0,0,True,False,False,False,True,False,False,False,False,False,False,True,False
4,14,2,2.0,0,1,True,False,False,False,False,False,False,False,True,False,False,True,False


In [169]:
# Check shape before and after encoding
print(f"Shape before encoding: {X.shape}")
print(f"Shape after encoding: {X_encoded.shape}")

Shape before encoding: (119210, 9)
Shape after encoding: (119210, 18)


After encoding, categorical columns were converted into dummy variables. The number of columns increased because each categorical feature was split into True/False columns. `drop_first=True` was used to avoid creating unnecessary duplicate dummy columns.

In [170]:
# Display all encoded feature names
print("Encoded feature names:")
print(X_encoded.columns.tolist())

Encoded feature names:
['lead_time', 'total_nights', 'total_guests', 'has_children', 'has_special_request', 'hotel_Resort Hotel', 'deposit_type_Non Refund', 'deposit_type_Refundable', 'market_segment_Complementary', 'market_segment_Corporate', 'market_segment_Direct', 'market_segment_Groups', 'market_segment_Offline TA/TO', 'market_segment_Online TA', 'market_segment_Undefined', 'customer_type_Group', 'customer_type_Transient', 'customer_type_Transient-Party']


The encoded column names were checked to confirm that the selected categorical features were converted successfully and that the dataset is ready for train-test split.

## 3.5 Train-Test Split

In [171]:
# Split data into train set and test set
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=2026)

In [172]:
# check the shape of the train and test sets
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")    
print(f"y_test shape: {y_test.shape}")

X_train shape: (95368, 18)
X_test shape: (23842, 18)
y_train shape: (95368,)
y_test shape: (23842,)


The dataset was split into 80% training data and 20% testing data. `random_state=2026` was used so that the same split can be reprocuced when the notebook is run again.

## 3.6 Feature Scaling

In [173]:
# Apply feature scaling


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [174]:
# Check scaled data shapes
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

X_train_scaled shape: (95368, 18)
X_test_scaled shape: (23842, 18)


Feature scaling was applied successfully. Scaling is useful for Logistic Regression because it is sensitive to the magnitude of features. Thus, scaling ensures that distance-based calculations treat all features equally. For example, an obvious difference in scale is seen in a huge `lead_time` vs a tiny number of `babies`. Tree-based models such as Decision Tree, Random Forest and Gradient Boosting do not strictly require scaling.

## 4. Modelling

## 4.1 Baseline Model: Logistic Regression

Logistic Regression is used as the baseline model because it is a simple and interpretable classification model. Its performance provides a starting point for comparing more complex models such as Decision Tree, Random Forest and Gradient Boosting.

In [175]:
# Train baseline model using Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=2026)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)

In [176]:
# Evaluate Logistic Regression model
print("Logistic Regression Model Evaluation:")
print("Accuracy:", accuracy_score(y_test, lr_pred))
print("Precision:", precision_score(y_test, lr_pred))
print("Recall:", recall_score(y_test, lr_pred))
print("F1-Score:", f1_score(y_test, lr_pred))

Logistic Regression Model Evaluation:
Accuracy: 0.7853368006039761
Precision: 0.8222870879120879
Recall: 0.5397880973850315
F1-Score: 0.6517419706042461


The Logistic Regression results were recorded as the baseline performance for comparison with other models.

## 4.2 Train Other Machine Learning Models

In [177]:
# Create and train other models for comparison

# Decision Tree Classifier
dt_model = DecisionTreeClassifier(random_state=2026)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

# Random Forest Classifier
rf_model = RandomForestClassifier(random_state=2026)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

# Gradient Boosting Classifier
gb_model = GradientBoostingClassifier(random_state=2026)
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)

In [178]:
# Evaluate the other models
print("\nDecision Tree Model Evaluation:")
print("Accuracy:", accuracy_score(y_test, dt_pred))
print("Precision:", precision_score(y_test, dt_pred))
print("Recall:", recall_score(y_test, dt_pred))
print("F1-Score:", f1_score(y_test, dt_pred))

print("\nRandom Forest Model Evaluation:")
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall:", recall_score(y_test, rf_pred))
print("F1-Score:", f1_score(y_test, rf_pred))

print("\nGradient Boosting Model Evaluation:")
print("Accuracy:", accuracy_score(y_test, gb_pred))
print("Precision:", precision_score(y_test, gb_pred))
print("Recall:", recall_score(y_test, gb_pred))
print("F1-Score:", f1_score(y_test, gb_pred))


Decision Tree Model Evaluation:
Accuracy: 0.8002264910661857
Precision: 0.7686674512880868
Recall: 0.6625338142470695
F1-Score: 0.7116653550457049

Random Forest Model Evaluation:
Accuracy: 0.8041271705393843
Precision: 0.7665567114945445
Recall: 0.6810189359783588
F1-Score: 0.7212605944848991

Gradient Boosting Model Evaluation:
Accuracy: 0.7962419260129184
Precision: 0.8140845070422535
Recall: 0.5863390441839496
F1-Score: 0.681693093958852


The models were compared using accuracy, precision, recall and F1-score. Since the target variable is imbalanced, accuracy alone may not give a complete picture of model performance.
F1-score is useful for this project because it balances precision and recall. This is important when creating hotel booking cancellations, as the model should identify cancelled bookings while also avoiding too many incorrect cancellation predictions.

In [179]:
# Compare models using F1-score
f1_results = {
    "Logistic Regression": f1_score(y_test, lr_pred),
    "Decision Tree": f1_score(y_test, dt_pred),
    "Random Forest": f1_score(y_test, rf_pred),
    "Gradient Boosting": f1_score(y_test, gb_pred)
}

f1_results

{'Logistic Regression': 0.6517419706042461,
 'Decision Tree': 0.7116653550457049,
 'Random Forest': 0.7212605944848991,
 'Gradient Boosting': 0.681693093958852}

In [180]:
# Find model with highest F1-score
best_model_name = max(f1_results, key=f1_results.get)
best_f1_score = f1_results[best_model_name]

print("Best model:", best_model_name)
print("Best F1-score:", best_f1_score)

Best model: Random Forest
Best F1-score: 0.7212605944848991


Based on the F1-score comparison, the best performing model is Random Forest. This model will be evaluated further using a Confusion Matrix and a Classification Report.

## 4.3 Hyperparameter Tuning

In [181]:
# Hyperparameter tuning for Random Forest using RandomizedSearchCV
rf_param_grid = {
    "n_estimators": [100, 200, 300], # Number of trees in the forest
    "max_depth": [5, 10, 20], # Maximum depth of the tree
    "min_samples_split": [2, 5, 10] # Minimum number of samples required to split an internal node
}
rf_random_search = RandomizedSearchCV(
    estimator=rf_model, 
    param_distributions=rf_param_grid, 
    n_iter=10, 
    cv=3,
    scoring='f1', 
    random_state=2026
    )
rf_random_search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...om_state=2026)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'max_depth': [5, 10, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [100, 200, ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default 

In [182]:
# Display best hyperparameters for Random Forest
print("Best hyperparameters for Random Forest:")
print(rf_random_search.best_params_)

Best hyperparameters for Random Forest:
{'n_estimators': 300, 'min_samples_split': 10, 'max_depth': 20}


RandomizedSearchCV selected the following best hyperparameters:
- `n_estimators`: 300
- `min_samples_split`: 10
- `max_depth`: 20

These values were selected because they gave the best F1-score during the hyperparameter search. The tuned model will then be evaluated on the test set to check whether the tuning improved the model performance.

In [183]:
# Evaluate tuned model on test set

tuned_model = rf_random_search.best_estimator_
tuned_pred = tuned_model.predict(X_test)

In [184]:
# Compare Random Forest before and after tuning

tuning_comparison = pd.DataFrame([
    {
        "Model": "Random Forest before tuning",
        "Accuracy": accuracy_score(y_test, rf_pred),
        "Precision": precision_score(y_test, rf_pred, zero_division=0),
        "Recall": recall_score(y_test, rf_pred, zero_division=0),
        "F1-score": f1_score(y_test, rf_pred, zero_division=0)
    },
    {
        "Model": "Random Forest after tuning",
        "Accuracy": accuracy_score(y_test, tuned_pred),
        "Precision": precision_score(y_test, tuned_pred, zero_division=0),
        "Recall": recall_score(y_test, tuned_pred, zero_division=0),
        "F1-score": f1_score(y_test, tuned_pred, zero_division=0)
    }
])

tuning_comparison

,Model,Accuracy,Precision,Recall,F1-score
0,Random Forest before tuning,0.804127,0.766557,0.681019,0.721261
1,Random Forest after tuning,0.816500,0.813118,0.658138,0.727465


The comparison table shows the Random Forest model performance before and after hyperparameter tuning.

After tuning, the F1-score improved from about 0.7213 to 0.7275. Accuracy and precision also improved, while recall decreased slightly. Since F1-score was used as the main metric, the tuned Random Forest model was selected as the final model.

The final hyperparameters selected were `n_estimators=300`, `min_samples_split=10` and `max_depth=20`. A reproducible setup was used by setting `random_state=2026`.

## 5. Model Evaluation

## 5.1 Confusion Matrix

In [188]:
# Create a confusion matrix for the tuned Random Forest model
conf_matrix = confusion_matrix(y_test, tuned_pred)
print("Confusion Matrix for Tuned Random Forest Model:")
print(conf_matrix)

Confusion Matrix for Tuned Random Forest Model:
[[13628  1342]
 [ 3033  5839]]


In [189]:
# Display confusion matrix as a table

conf_matrix_df = pd.DataFrame(conf_matrix, index=['Actual Not Cancelled', 'Actual Cancelled'], columns=['Predicted Not Cancelled', 'Predicted Cancelled'])
print("Confusion Matrix for Tuned Random Forest Model:")
print(conf_matrix_df)

Confusion Matrix for Tuned Random Forest Model:
                      Predicted Not Cancelled  Predicted Cancelled
Actual Not Cancelled                    13628                 1342
Actual Cancelled                         3033                 5839


The confusion matrix shows the number of correct and incorrect predictions made by the tuned Random Forest model.

The model correctly predicted 13,628 not cancelled bookings and 5,839 cancelled bookings. However, it wrongly predicted 1,342 not cancelled bookings as cancelled and missed 3,033 actual cancelled bookings by predicting them as not cancelled.

This shows that the model performs well overall, but it misses some cancelled bookings.

## 5.2 Classification Report

In [192]:
# Classification report for the tuned Random Forest model
class_report = classification_report(y_test, tuned_pred, target_names=['Not Cancelled', 'Cancelled'])
print("Classification Report for Tuned Random Forest Model:")
print(class_report)

Classification Report for Tuned Random Forest Model:
               precision    recall  f1-score   support

Not Cancelled       0.82      0.91      0.86     14970
    Cancelled       0.81      0.66      0.73      8872

     accuracy                           0.82     23842
    macro avg       0.82      0.78      0.79     23842
 weighted avg       0.82      0.82      0.81     23842



The classification report shows the precision, recall and F1-score for both booking outcomes.

For the not cancelled class, the model achieved a precision of 0.82, recall of 0.91 and F1-score of 0.86. This shows that the model is strong in identifying bookings that are not cancelled.

For the cancelled class, the model achieved a precision of 0.81, recall of 0.66 and F1-score of 0.73. This means that when the model predicts a booking as cancelled, it is usually correct, but it does not identify all actual cancellations.

The overall accuracy is 0.82, which means the model correctly predicted about 82% of the test bookings.

## 5.3 Evaluation Summary

The tuned Random Forest model performed reasonably well on the test set. It achieved good overall accuracy and a balanced F1-score, which is important because the target variable is imbalanced.

The model is better at predicting not cancelled bookings than cancelled bookings. However, it still identifies many cancelled bookings correctly and has a cancelled-class F1-score of 0.73.

Overall, the tuned Random Forest model is suitable for the project because it provides useful cancellation predictions and can support the Streamlit application by giving users a predicted cancellation outcome.

## 6. Iterative model development
